[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C16_Generative_Models_Course/03_gan/03_gan.ipynb)

# 03 · 生成对抗网络 GAN（用 numpy 从零）

目标：从零实现一个 **1D GAN**（生成器 G、判别器 D、两套损失、交替更新、手写反向），在玩具双峰分布上**收敛到目标分布**；演示 **模式坍塌** 与稳定技巧。

路线：最优判别器(对拍闭式) → D/G 两套损失 → **梯度穿过 D 更新 G** → 交替训练收敛 → 模式坍塌 → ✏️ 练习（D loss、G loss、交替更新、坍塌演示）→ 📖 答案 → 🧪 真实(非饱和损失)胶囊。

> 心智模型：**G 造假、D 验真，对抗到 D 处处输出 1/2、p_g=p_data。没有可信的单一损失——只能看生成分布**。

## 1 · 最优判别器 D* = p_data/(p_data+p_g)

固定 G 时，最优判别器有闭式 `D*(x)=p_data(x)/(p_data(x)+p_g(x))`。

我们用两个**已知**的高斯分布当 `p_data`、`p_g`，直接算出 `D*`，并验证：在 `p_data` 占优处 `D*->1`，相等处 `D*=0.5`。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def gauss_pdf(x, mu, sig):
    return np.exp(-0.5*((x-mu)/sig)**2) / (sig*np.sqrt(2*np.pi))

def optimal_D(p_data, p_g):
    return p_data / (p_data + p_g + 1e-12)

xs = np.linspace(-5, 5, 200)
pd = gauss_pdf(xs, 1.0, 1.0)      # 真实：N(1,1)
pg = gauss_pdf(xs, -1.0, 1.0)     # 生成：N(-1,1)
Dstar = optimal_D(pd, pg)
# 在 x>>0（真实占优）D*->1；x<<0（生成占优）D*->0；两分布相等处(x=0) D*=0.5
x3 = np.array([3.0])
print('D*(x=3)  = %.3f  (真实占优 -> ~1)' % optimal_D(gauss_pdf(x3, 1, 1), gauss_pdf(x3, -1, 1))[0])
i0 = np.argmin(np.abs(xs))        # x≈0
print('D*(x≈0)  = %.3f  (两分布对称相等 -> 0.5)' % Dstar[i0])
assert abs(Dstar[i0] - 0.5) < 0.02, '对称点处 D*=0.5'
assert Dstar[xs > 3].mean() > 0.9, '真实占优处 D*->1'
assert Dstar[xs < -3].mean() < 0.1, '生成占优处 D*->0'
print('✅ 最优判别器 = 真假密度比；相等处只能瞎猜(0.5) —— 这是纳什均衡的样子')

## 2 · 搭建 G、D 与判别器损失

`G: z(1)->tanh->x(1)`，`D: x(1)->tanh->sigmoid 概率`。

判别器损失（二分类交叉熵，要最小化）：`L_D = -mean log D(x_real) - mean log(1-D(x_fake))`，让真->1、假->0。

In [ ]:
def sigmoid(x): return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def init_gan(seed=0, H=32, s=0.8):
    g = np.random.default_rng(seed)
    return dict(
        Gw1=s*g.standard_normal((1,H)), Gb1=np.zeros(H), Gw2=s*g.standard_normal((H,1)), Gb2=np.zeros(1),
        Dw1=s*g.standard_normal((1,H)), Db1=np.zeros(H), Dw2=s*g.standard_normal((H,1)), Db2=np.zeros(1))

def G_forward(z, P):
    h = np.tanh(z @ P['Gw1'] + P['Gb1'])
    return h @ P['Gw2'] + P['Gb2'], h

def D_forward(x, P):
    h = np.tanh(x @ P['Dw1'] + P['Db1'])
    prob = sigmoid(h @ P['Dw2'] + P['Db2'])
    return prob, h

def real_data(n, g, sep=2.0, noise=0.3):
    comp = g.integers(0, 2, n)
    return (np.where(comp==0, -sep, sep) + noise*g.standard_normal(n)).reshape(-1, 1)

def D_loss(P, x_real, x_fake):
    dr, _ = D_forward(x_real, P); df, _ = D_forward(x_fake, P)
    eps = 1e-7
    return -np.mean(np.log(dr+eps)) - np.mean(np.log(1-df+eps))

P = init_gan(); g = np.random.default_rng(1)
xr = real_data(128, g); z = g.standard_normal((128,1)); xf, _ = G_forward(z, P)
print('初始 D(真)均值=%.3f  D(假)均值=%.3f  L_D=%.3f' % (D_forward(xr,P)[0].mean(), D_forward(xf,P)[0].mean(), D_loss(P,xr,xf)))
assert D_loss(P, xr, xf) > 0
print('✅ G、D 网络与判别器损失就位')

## 3 · 两套梯度：D 的梯度 + G 的梯度（穿过 D）

**判别器梯度**：sigmoid+CE 下，真样本 `dlogit=(D(x)-1)/N`、假样本 `dlogit=D(x)/N`。

**生成器梯度**（非饱和 `L_G=-mean log D(G(z))`）：`dlogit_fake=(D(G(z))-1)/N`，然后**梯度穿过 D**（D 参数不动，只借通路）传到 `x_fake`、再传到 G。用数值梯度检验守住。

In [ ]:
def D_grads(P, x_real, x_fake):
    '''只返回 D 参数的梯度（G 视为固定）。'''
    N = x_real.shape[0]
    dr, hr = D_forward(x_real, P); df, hf = D_forward(x_fake, P)
    dlr = (dr - 1) / N           # 真样本想被判为1
    dlf = df / N                 # 假样本想被判为0
    g = {}
    g['Dw2'] = hr.T @ dlr + hf.T @ dlf; g['Db2'] = (dlr + dlf).sum(0)
    dhr = (dlr @ P['Dw2'].T) * (1 - hr**2); dhf = (dlf @ P['Dw2'].T) * (1 - hf**2)
    g['Dw1'] = x_real.T @ dhr + x_fake.T @ dhf; g['Db1'] = (dhr + dhf).sum(0)
    return g

def G_grads(P, z):
    '''生成器梯度：穿过 D（D 参数不更新）传回 G。非饱和损失。'''
    N = z.shape[0]
    x_fake, hg = G_forward(z, P)
    df, hf = D_forward(x_fake, P)
    dlf = (df - 1) / N                       # 非饱和: 让 D(G(z))->1
    dhf = (dlf @ P['Dw2'].T) * (1 - hf**2)
    dx_fake = dhf @ P['Dw1'].T               # 梯度到达 x_fake（穿过 D）
    g = {}
    g['Gw2'] = hg.T @ dx_fake; g['Gb2'] = dx_fake.sum(0)
    dhg = (dx_fake @ P['Gw2'].T) * (1 - hg**2)
    g['Gw1'] = z.T @ dhg; g['Gb1'] = dhg.sum(0)
    return g

# 数值梯度检验 D（固定 xr, xf）
P = init_gan(); g = np.random.default_rng(2)
xr = real_data(64, g); z = g.standard_normal((64,1)); xf, _ = G_forward(z, P)
gd = D_grads(P, xr, xf)
def check(P, key, lossfn):
    W = P[key]; gn = np.zeros_like(W); it = np.nditer(W, flags=['multi_index'])
    while not it.finished:
        i = it.multi_index; o = W[i]
        W[i]=o+1e-6; fp=lossfn(); W[i]=o-1e-6; fm=lossfn(); W[i]=o
        gn[i]=(fp-fm)/2e-6; it.iternext()
    return np.max(np.abs(gn - 0))  # placeholder
# 检验 Dw1
W = P['Dw1']; gn = np.zeros_like(W); it = np.nditer(W, flags=['multi_index'])
while not it.finished:
    i=it.multi_index; o=W[i]; W[i]=o+1e-6; fp=D_loss(P,xr,xf); W[i]=o-1e-6; fm=D_loss(P,xr,xf); W[i]=o
    gn[i]=(fp-fm)/2e-6; it.iternext()
assert np.max(np.abs(gn - gd['Dw1'])) < 1e-5, 'D 梯度应与数值一致'
# 检验 G 梯度（固定 D，损失=-mean log D(G(z))）
def G_loss(P, z):
    xf, _ = G_forward(z, P); df, _ = D_forward(xf, P); return -np.mean(np.log(df+1e-7))
gg = G_grads(P, z)
W = P['Gw1']; gn = np.zeros_like(W); it = np.nditer(W, flags=['multi_index'])
while not it.finished:
    i=it.multi_index; o=W[i]; W[i]=o+1e-6; fp=G_loss(P,z); W[i]=o-1e-6; fm=G_loss(P,z); W[i]=o
    gn[i]=(fp-fm)/2e-6; it.iternext()
assert np.max(np.abs(gn - gg['Gw1'])) < 1e-5, 'G 梯度(穿过D)应与数值一致'
print('✅ 两套梯度都对：D 的梯度、G 的梯度(穿过 D 传回)，均通过数值检验')

## 4 · 交替训练：在博弈中收敛到双峰分布

每步：先更新 D（用真+假），再更新 G（穿过 D）。训练后看生成分布是否收敛到目标双峰。

**注意**：GAN 没有可信的单一损失，我们直接看**生成分布的统计**——均值、标准差、是否双峰、是否覆盖两个模式。

In [ ]:
def train_gan(epochs=10000, lr=0.03, seed=0, H=32, bs=256, sep=2.0):
    P = init_gan(seed=seed, H=H); g = np.random.default_rng(7)
    for ep in range(epochs):
        # --- 更新 D ---
        xr = real_data(bs, g, sep=sep); z = g.standard_normal((bs,1)); xf, _ = G_forward(z, P)
        gd = D_grads(P, xr, xf)
        for k in ['Dw1','Db1','Dw2','Db2']: P[k] -= lr * gd[k]
        # --- 更新 G（重采 z，梯度穿过 D，但只更新 G）---
        z = g.standard_normal((bs,1))
        gg = G_grads(P, z)
        for k in ['Gw1','Gb1','Gw2','Gb2']: P[k] -= lr * gg[k]
    return P

P_t = train_gan(epochs=10000, lr=0.03, seed=0)
g = np.random.default_rng(99)
real = real_data(3000, g, sep=2.0).ravel()
z = g.standard_normal((3000,1)); fake, _ = G_forward(z, P_t); fake = fake.ravel()
print('真实  均值=%.2f std=%.2f' % (real.mean(), real.std()))
print('生成  均值=%.2f std=%.2f' % (fake.mean(), fake.std()))
print('生成落在两峰之间谷底 |x|<1 的比例 = %.3f (真实≈%.3f)' % (np.mean(np.abs(fake)<1), np.mean(np.abs(real)<1)))
frac_neg = np.mean(fake < 0)
print('生成落在左峰(x<0)的比例 = %.2f (均衡覆盖两峰 -> 接近0.5)' % frac_neg)
# 收敛判据：均值≈0、std≈真实、双峰(谷底稀疏)、两峰均衡(未坍塌)
assert abs(fake.mean()) < 0.4, '生成均值应接近 0（覆盖对称双峰）'
assert abs(fake.std() - real.std()) < 0.5, '生成 spread 应接近真实'
assert np.mean(np.abs(fake)<1) < 0.20, '应学出双峰(谷底稀疏)'
assert 0.30 < frac_neg < 0.70, '应均衡覆盖两个峰(未模式坍塌)'
print('✅ GAN 收敛：生成分布逼近目标双峰、均衡覆盖两峰 —— 全靠对抗信号，无似然！')

## 5 · 收敛的标志：判别器被骗懵（D≈0.5）

理论上 GAN 收敛时 `p_g≈p_data`，判别器无法分辨真假，输出趋近 **0.5**。
比较训练后 D 对真/假样本的平均输出——应该都接近 0.5（而非初始时真假分得很开）。

In [ ]:
d_real = D_forward(real.reshape(-1,1), P_t)[0].mean()
d_fake = D_forward(fake.reshape(-1,1), P_t)[0].mean()
print('训练后 D(真)均值 = %.3f' % d_real)
print('训练后 D(假)均值 = %.3f' % d_fake)
print('两者之差 = %.3f (越接近 0 说明 D 越分不出真假 -> 越收敛)' % abs(d_real - d_fake))
# D 对真假的判断应该接近(都在0.5附近)，说明生成已逼真
assert abs(d_real - d_fake) < 0.25, 'D 对真假的输出应接近(被骗懵)'
assert 0.3 < d_fake < 0.7, 'D(假) 应在 0.5 附近'
print('✅ 判别器被骗到接近瞎猜(≈0.5) —— 这正是纳什均衡的实证信号')

## 6 · 模式坍塌：生成器只覆盖部分模式

模式坍塌是 GAN 头号顽疾。我们人为制造它：用一个**容量极小**、且训练**极不平衡**（G 学太快/D 太弱）的设置，让生成器只学会产出一个峰，丢掉另一个。对比正常训练，看「只覆盖单峰」。

In [ ]:
def train_gan_collapse(epochs=8000, seed=3):
    '''刻意诱发坍塌：极小 G 容量 + 让 G 每步多更新几次(压过 D)。'''
    P = init_gan(seed=seed, H=4, s=1.5); g = np.random.default_rng(7)   # H=4 容量很小
    for ep in range(epochs):
        xr = real_data(256, g, sep=2.0); z = g.standard_normal((256,1)); xf,_ = G_forward(z, P)
        gd = D_grads(P, xr, xf)
        for k in ['Dw1','Db1','Dw2','Db2']: P[k] -= 0.005 * gd[k]   # D 学得慢(弱)
        for _ in range(4):                                          # G 每步多更新(强)
            z = g.standard_normal((256,1)); gg = G_grads(P, z)
            for k in ['Gw1','Gb1','Gw2','Gb2']: P[k] -= 0.05 * gg[k]
    return P

P_c = train_gan_collapse()
g = np.random.default_rng(99); z = g.standard_normal((3000,1))
fake_c, _ = G_forward(z, P_c); fake_c = fake_c.ravel()
frac_neg_c = np.mean(fake_c < 0)
print('坍塌模型: 生成落左峰比例=%.2f, 落右峰比例=%.2f' % (frac_neg_c, 1-frac_neg_c))
print('对比正常模型: 落左峰比例=%.2f' % np.mean(fake < 0))
# 坍塌：几乎全挤在一个峰（frac_neg 接近 0 或 1），远离均衡的 0.5
assert abs(frac_neg_c - 0.5) > abs(np.mean(fake < 0) - 0.5), '坍塌模型比正常模型更偏向单峰'
print('✅ 模式坍塌复现：失衡训练 + 小容量 -> 生成器丢掉一个峰、只覆盖部分模式')

---
## ✏️ 练习 1：判别器损失

实现 `disc_loss(d_real, d_fake)`：给定 D 对真/假样本输出的概率，返回二分类交叉熵 `-mean log(d_real) - mean log(1-d_fake)`。用 `eps` 防 log(0)。

In [ ]:
def disc_loss(d_real, d_fake, eps=1e-7):
    # TODO: -mean log(d_real) - mean log(1 - d_fake)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 完美判别器：真->1, 假->0 -> 损失≈0
dr = np.full(50, 1-1e-7); df = np.full(50, 1e-7)
assert disc_loss(dr, df) < 1e-5
# 瞎猜：真假都判 0.5 -> 损失 = -2*log(0.5) = 2log2
assert abs(disc_loss(np.full(50,0.5), np.full(50,0.5)) - 2*np.log(2)) < 1e-5
print('✅ 练习 1 通过：判别器损失正确（完美->0，瞎猜->2log2）')

## ✏️ 练习 2：生成器损失（饱和 vs 非饱和）

实现两种 G 损失：
(a) `g_loss_saturating(d_fake)` = `mean log(1-d_fake)`（原始 minimax，G 想最小化）；
(b) `g_loss_nonsat(d_fake)` = `-mean log(d_fake)`（非饱和，G 想最小化）。
二者都让 `d_fake->1`，但非饱和在 `d_fake≈0` 时梯度更大。

In [ ]:
def g_loss_saturating(d_fake, eps=1e-7):
    # TODO: mean log(1 - d_fake)
    raise NotImplementedError

def g_loss_nonsat(d_fake, eps=1e-7):
    # TODO: -mean log(d_fake)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# d_fake 很小(G 很烂, D 识破)时，非饱和损失对 d_fake 的梯度幅度应远大于饱和损失
df_bad = 0.01
h = 1e-6
grad_sat = abs((g_loss_saturating(np.array([df_bad+h])) - g_loss_saturating(np.array([df_bad-h])))/(2*h))
grad_ns  = abs((g_loss_nonsat(np.array([df_bad+h])) - g_loss_nonsat(np.array([df_bad-h])))/(2*h))
print('d_fake=0.01 处梯度幅度: 饱和=%.2f  非饱和=%.2f' % (grad_sat, grad_ns))
assert grad_ns > grad_sat, '非饱和损失在烂生成器处梯度更强(不饱和)'
# 两者方向一致：d_fake 增大都使损失减小
assert g_loss_nonsat(np.array([0.9])) < g_loss_nonsat(np.array([0.1]))
print('✅ 练习 2 通过：非饱和损失在烂生成器处给更强梯度（救命的工程改动）')

## ✏️ 练习 3：一步交替更新

实现 `gan_step(P, real_batch, z_batch, lr)`：执行**一步**完整交替——先用 `D_grads` 更新 D 的 4 个参数，再用 `G_grads` 更新 G 的 4 个参数（**只更新各自的参数**）。返回更新后的 `P`。

In [ ]:
def gan_step(P, real_batch, z_batch, lr=0.03):
    # TODO: x_fake = G_forward(z_batch,P)[0]; 用 D_grads 更新 Dw1,Db1,Dw2,Db2
    #       再用 G_grads(P, z_batch) 更新 Gw1,Gb1,Gw2,Gb2；返回 P
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
P = init_gan(seed=5); g = np.random.default_rng(11)
Dw1_before = P['Dw1'].copy(); Gw1_before = P['Gw1'].copy()
xr = real_data(128, g); zb = g.standard_normal((128,1))
P = gan_step(P, xr, zb, lr=0.03)
assert not np.allclose(P['Dw1'], Dw1_before), 'D 参数应被更新'
assert not np.allclose(P['Gw1'], Gw1_before), 'G 参数应被更新'
# 多跑几步应降低 D 对真假的混淆度(初期)，这里只验证能稳定迭代不发散
for _ in range(200):
    xr = real_data(128, g); zb = g.standard_normal((128,1)); P = gan_step(P, xr, zb)
xf, _ = G_forward(g.standard_normal((500,1)), P)
assert np.all(np.isfinite(xf)), '训练应数值稳定(不 NaN/inf)'
print('✅ 练习 3 通过：一步交替更新正确、可稳定迭代')

## ✏️ 练习 4：检测模式坍塌

实现 `mode_coverage(samples, sep=2.0)`：返回生成样本对**两个峰**（中心 ±sep）的覆盖比例 `(左峰比例, 右峰比例)`（按离哪个峰近划分）。坍塌时一个比例会接近 0。

In [ ]:
def mode_coverage(samples, sep=2.0):
    # TODO: 按样本离 -sep 还是 +sep 近，分到左/右峰；返回 (左比例, 右比例)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
# 均衡覆盖的样本
balanced = np.concatenate([-2+0.3*rng.standard_normal(500), 2+0.3*rng.standard_normal(500)])
l, r = mode_coverage(balanced)
assert abs(l - 0.5) < 0.1 and abs(r - 0.5) < 0.1 and abs(l+r-1) < 1e-9
# 坍塌样本：全在右峰
collapsed = 2 + 0.3*rng.standard_normal(1000)
l2, r2 = mode_coverage(collapsed)
assert l2 < 0.05 and r2 > 0.95, '坍塌到右峰 -> 左峰覆盖≈0'
print('✅ 练习 4 通过：能检测模式覆盖/坍塌（左 %.2f 右 %.2f vs 坍塌 左 %.2f 右 %.2f）' % (l,r,l2,r2))

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def disc_loss(d_real, d_fake, eps=1e-7):
    return -np.mean(np.log(d_real + eps)) - np.mean(np.log(1 - d_fake + eps))

In [ ]:
# 练习 2 参考答案
def g_loss_saturating(d_fake, eps=1e-7):
    return np.mean(np.log(1 - d_fake + eps))

def g_loss_nonsat(d_fake, eps=1e-7):
    return -np.mean(np.log(d_fake + eps))

In [ ]:
# 练习 3 参考答案
def gan_step(P, real_batch, z_batch, lr=0.03):
    x_fake, _ = G_forward(z_batch, P)
    gd = D_grads(P, real_batch, x_fake)
    for k in ['Dw1','Db1','Dw2','Db2']: P[k] -= lr * gd[k]
    gg = G_grads(P, z_batch)
    for k in ['Gw1','Gb1','Gw2','Gb2']: P[k] -= lr * gg[k]
    return P

In [ ]:
# 练习 4 参考答案
def mode_coverage(samples, sep=2.0):
    s = np.asarray(samples).ravel()
    left = np.mean(np.abs(s - (-sep)) < np.abs(s - sep))
    return float(left), float(1 - left)

---
## 🧪 真实技巧胶囊：非饱和损失为何是标准做法

Goodfellow 原论文就指出：原始 minimax 的 G 损失在训练初期**饱和**（梯度消失），改用非饱和损失救命。
这里**定量复现**：在一系列 `d_fake`（从烂到好）上，比较两种损失的梯度幅度。**纯本地计算**。

In [ ]:
# 在 d_fake 从 0.01(很烂) 到 0.5 上比较两种 G 损失的梯度幅度
d_vals = np.array([0.01, 0.05, 0.1, 0.3, 0.5])
h = 1e-6
print(f"{'d_fake':>8} {'饱和|grad|':>12} {'非饱和|grad|':>14} {'倍数':>8}")
ratios = []
for dv in d_vals:
    gs = abs((g_loss_saturating(np.array([dv+h])) - g_loss_saturating(np.array([dv-h])))/(2*h))
    gn = abs((g_loss_nonsat(np.array([dv+h])) - g_loss_nonsat(np.array([dv-h])))/(2*h))
    ratios.append(gn/gs)
    print(f'{dv:>8.2f} {gs:>12.3f} {gn:>14.3f} {gn/gs:>8.1f}x')
print('✅ 数据就绪：d_fake 越小(G越烂)，非饱和损失的梯度优势越大')

**🧪 胶囊练习**：实现 `nonsat_advantage_grows(d_vals, ratios)`：验证「`d_fake` 越小，非饱和/饱和的梯度比越大」——即把 `d_vals` 升序排列后，梯度比 `ratios` 应**单调不增**。返回 bool。

In [ ]:
def nonsat_advantage_grows(d_vals, ratios):
    # TODO: 按 d_vals 升序，检查 ratios 单调不增（容忍 1e-6）；返回 bool
    raise NotImplementedError

In [ ]:
# 自测
ok = nonsat_advantage_grows(d_vals, ratios)
assert ok, 'd_fake 越小，非饱和损失梯度优势越大'
# 最烂处(d_fake=0.01)的优势应非常大
assert ratios[0] > 10, '在 d_fake=0.01 处非饱和梯度应远大于饱和(>10x)'
print('✅ 胶囊练习通过：定量验证非饱和损失在烂生成器处的梯度优势')

In [ ]:
# 📖 胶囊参考答案
def nonsat_advantage_grows(d_vals, ratios):
    order = np.argsort(d_vals)
    r = np.array(ratios)[order]
    return bool(all(r[i+1] <= r[i] + 1e-6 for i in range(len(r)-1)))

### 小结
- GAN = G(造假) vs D(验真) 的 minimax 博弈；最优 D = 真假密度比，此时 G 在最小化 JS 散度。
- 两套损失、两套梯度：**G 的梯度要穿过 D**（D 不更新）；用**非饱和损失**避免初期梯度消失。
- 交替训练逼近目标分布；收敛时 D 被骗到 ≈0.5（瞎猜）。
- **模式坍塌**（只覆盖部分模式）与**训练不稳定**是两大顽疾；**GAN 损失不可信，必须看生成分布**。
- GAN(锐利但难训、易坍塌) vs VAE(模糊但稳、覆盖广)：一组镜像的权衡。

下一站：**模块 04 · 扩散 DDPM** —— 不用对抗、不写似然，把生成拆成几十步「预测噪声」，既稳又锐。